# OpenAI Whisper — Audio / Video Transcription

Based on [tommykho/whisper](https://github.com/tommykho/whisper).

Transcribes audio/video files to text locally using OpenAI's [Whisper](https://github.com/openai/whisper) model.
No API key required — runs entirely on your machine.

**Supported formats:** MP4, MOV, MKV, AVI, MP3, WAV, M4A, FLAC

**Models:** `tiny`, `base`, `small`, `medium`, `large`, `turbo`  
Larger models are slower but more accurate. `base` is a good starting point.

**Requirements:**
- `pip install openai-whisper torch`
- [FFmpeg](https://ffmpeg.org/) in PATH (for video/non-WAV audio extraction)

In [ ]:
# !pip install openai-whisper torch

In [ ]:
import whisper
import torch
import os
import subprocess
import time

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load model

In [ ]:
MODEL_NAME = "base"  # tiny | base | small | medium | large | turbo

print(f"Loading Whisper model: {MODEL_NAME} ...")
model = whisper.load_model(MODEL_NAME, device=device)
print("Model loaded.")

## Extract audio from video (FFmpeg)

In [ ]:
def extract_audio(input_path, output_path="audio_extracted.wav"):
    """Extract 16 kHz mono WAV from any audio/video file using FFmpeg."""
    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-ar", "16000",
        "-ac", "1",
        "-c:a", "pcm_s16le",
        output_path
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    print(f"Audio extracted to: {output_path}")
    return output_path

## Transcribe a file

In [ ]:
def transcribe(input_file, language=None):
    """
    Transcribe an audio/video file.
    
    Args:
        input_file: path to audio or video file
        language:   ISO language code (e.g. 'en', 'ja') or None for auto-detect
    Returns:
        Transcribed text string
    """
    ext = os.path.splitext(input_file)[1].lower()
    video_exts = {".mp4", ".mov", ".mkv", ".avi", ".webm"}

    # Extract audio from video files
    if ext in video_exts:
        audio_file = extract_audio(input_file)
    else:
        audio_file = input_file

    print(f"Transcribing: {audio_file} ...")
    start = time.time()

    kwargs = {}
    if language:
        kwargs["language"] = language

    result = model.transcribe(audio_file, **kwargs)
    elapsed = time.time() - start

    print(f"Done in {elapsed:.1f}s")
    return result["text"]


# ----- Example usage -----
# Set INPUT_FILE to your audio or video path, then run:
INPUT_FILE = "sample.mp3"   # replace with your file

if os.path.exists(INPUT_FILE):
    text = transcribe(INPUT_FILE)
    print("\n--- Transcription ---")
    print(text)
else:
    print(f"File not found: {INPUT_FILE}  — set INPUT_FILE to a valid path.")

## Save transcription to file

In [ ]:
def save_transcription(text, input_file):
    out_path = os.path.splitext(input_file)[0] + ".txt"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Saved: {out_path}")
    return out_path

# Uncomment to save:
# save_transcription(text, INPUT_FILE)

## Batch transcription

In [ ]:
import glob

# Transcribe all MP3 files in current directory
files = glob.glob("*.mp3") + glob.glob("*.wav") + glob.glob("*.mp4")
print(f"Found {len(files)} file(s): {files}")

for f in files:
    text = transcribe(f)
    save_transcription(text, f)